# Predicción de abandono de servicio en clientes de comercio electrónico.

# Nombre del autor: Eduardo Barahona Muñoz

# Preparación
El objetivo de esta tarea es realizar un análisis exploratorio de los datos y entrenar y evaluar modelos de machine learning, utilizando PySpark para el tratamiento de datos y scikit-learn para el entrenamiento y evaluación de los modelos.

Origen del dataset: https://www.kaggle.com/datasets/shriyashjagtap/e-commerce-customer-for-behavior-analysis

# Descripción

El conjunto de datos contiene las siguientes columnas: 

**ID del cliente**: Un identificador único para cada cliente. 

**Nombre del cliente**: El nombre del cliente 

**Edad del cliente**: La edad del cliente 

**Género**: El género del cliente

**Fecha de compra**: La fecha de cada compra realizada por el cliente. 

**Categoría del producto**: La categoría o tipo del producto adquirido. 

**Precio del producto**: El precio del producto adquirido. 

**Cantidad**: La cantidad del producto adquirido. 

**Importe total de la compra**: El importe total gastado por el cliente en cada transacción. 

**Método de pago**: El método de pago utilizado por el cliente (p. ej., tarjeta de crédito, PayPal). 

**Devoluciones**: Si el cliente devolvió algún producto del pedido (binario: 0 para no devolver, 1 para devolver). 

**Cancelación**: Una columna binaria que indica si el cliente ha cancelado su suscripción (0 para retenido, 1 para cancelado).


In [0]:
import re
# Cogemos nuesto usuario 
#user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().tags().apply('user')
user = spark.sql("SELECT current_user()").collect()[0][0]
userName = re.sub(r"[^a-zA-Z0-9]", "_", user)
# y creamos la bbdd con el nombre 
databaseName = userName + "_db"

spark.sql("CREATE DATABASE IF NOT EXISTS `{}`".format(databaseName))
# Le decimos a spark que vamos a usar nuestra bbdd
spark.sql("USE `{}`".format(databaseName))
print("Using database :::", databaseName)

Using database ::: barahonavilla_gmail_com_db


In [0]:
# Tras copiar nuestro fichero a la ruta de nuestro usuario comprobamos que está copiado correctamente
dataPath = "/Volumes/main/default/ecommerce_data/ecommerce_customer_data_custom_ratios.csv"
display(dbutils.fs.ls(dataPath))

path,name,size,modificationTime
dbfs:/Volumes/main/default/ecommerce_data/ecommerce_customer_data_custom_ratios.csv,ecommerce_customer_data_custom_ratios.csv,21418688,1775818822000


In [0]:
# Vamos a crear el dataframe con el que vamos a trabajar
ecomerceDF = (spark.read            
   .option("header", "true")       # Le decimos que el fichero lleva cabecera
   .option("inferSchema", "true")  # Y que asigne los tipos automáticamente
   .csv(dataPath)                  # path de nuestro fichero    
)

In [0]:
# Comprobamos que está bien creado
display(ecomerceDF.limit(10))

Customer ID,Purchase Date,Product Category,Product Price,Quantity,Total Purchase Amount,Payment Method,Customer Age,Returns,Customer Name,Age,Gender,Churn
46251,2020-09-08T09:38:32.000Z,Electronics,12,3,740,Credit Card,37,0.0,Christine Hernandez,37,Male,0
46251,2022-03-05T12:56:35.000Z,Home,468,4,2739,PayPal,37,0.0,Christine Hernandez,37,Male,0
46251,2022-05-23T18:18:01.000Z,Home,288,2,3196,PayPal,37,0.0,Christine Hernandez,37,Male,0
46251,2020-11-12T13:13:29.000Z,Clothing,196,1,3509,PayPal,37,0.0,Christine Hernandez,37,Male,0
13593,2020-11-27T17:55:11.000Z,Home,449,1,3452,Credit Card,49,0.0,James Grant,49,Female,1
13593,2023-03-07T14:17:42.000Z,Home,250,4,575,PayPal,49,1.0,James Grant,49,Female,1
13593,2023-04-15T03:02:33.000Z,Electronics,73,1,1896,Credit Card,49,0.0,James Grant,49,Female,1
13593,2021-03-27T21:23:28.000Z,Books,337,2,2937,Cash,49,0.0,James Grant,49,Female,1
13593,2020-05-05T20:14:00.000Z,Clothing,182,2,3363,PayPal,49,1.0,James Grant,49,Female,1
28805,2023-09-13T04:24:00.000Z,Electronics,394,2,1993,Credit Card,19,0.0,Jose Collier,19,Male,0


In [0]:
# Vamos a generar una tabla en nuestra bbdd con formato deltalake llamada ecomerce
# Primero debemos formatear los nombres de nuestras columnas para que sean válidos para nuestra tabla
from pyspark.sql.functions import col

# Renombrar columnas para asegurarse de que no haya caracteres inválidos
cleaned_columns = [col(column).alias(column.replace(" ", "_").replace(",", "_").replace(";", "_").replace(":", "_")) for column in ecomerceDF.columns]
ecomerceDF_table = ecomerceDF.select(*cleaned_columns)
ecomerceDF_table.write.mode("overwrite").format("delta").saveAsTable("ecomerce")

In [0]:
%sql
SELECT * FROM ecomerce LIMIT 10

Customer_ID,Purchase_Date,Product_Category,Product_Price,Quantity,Total_Purchase_Amount,Payment_Method,Customer_Age,Returns,Customer_Name,Age,Gender,Churn
46251,2020-09-08T09:38:32.000Z,Electronics,12,3,740,Credit Card,37,0.0,Christine Hernandez,37,Male,0
46251,2022-03-05T12:56:35.000Z,Home,468,4,2739,PayPal,37,0.0,Christine Hernandez,37,Male,0
46251,2022-05-23T18:18:01.000Z,Home,288,2,3196,PayPal,37,0.0,Christine Hernandez,37,Male,0
46251,2020-11-12T13:13:29.000Z,Clothing,196,1,3509,PayPal,37,0.0,Christine Hernandez,37,Male,0
13593,2020-11-27T17:55:11.000Z,Home,449,1,3452,Credit Card,49,0.0,James Grant,49,Female,1
13593,2023-03-07T14:17:42.000Z,Home,250,4,575,PayPal,49,1.0,James Grant,49,Female,1
13593,2023-04-15T03:02:33.000Z,Electronics,73,1,1896,Credit Card,49,0.0,James Grant,49,Female,1
13593,2021-03-27T21:23:28.000Z,Books,337,2,2937,Cash,49,0.0,James Grant,49,Female,1
13593,2020-05-05T20:14:00.000Z,Clothing,182,2,3363,PayPal,49,1.0,James Grant,49,Female,1
28805,2023-09-13T04:24:00.000Z,Electronics,394,2,1993,Credit Card,19,0.0,Jose Collier,19,Male,0


In [0]:
# Comprobamos los tipos de datos
ecomerceDF.printSchema()

root
 |-- Customer ID: integer (nullable = true)
 |-- Purchase Date: timestamp (nullable = true)
 |-- Product Category: string (nullable = true)
 |-- Product Price: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Total Purchase Amount: integer (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Customer Age: integer (nullable = true)
 |-- Returns: double (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Churn: integer (nullable = true)



# Análisis exploratorio de datos
Como podemos observar en el fichero tenemos una variable objetivo que es 'Churn', contendrá un 0 si el cliente permanece y 1 si abandona el servicio.

Tenemos un total de 250.000 registros. De los cuales 49.874 han abandonado el servicio y 200.126 han permanecido

Hemos revisado algunos datos más abajo y no se encuentra una relación directa entre alguno de los campos como el género, la cantidad de devoluciones, edad o categoría de producto con que haya permanecido o abandonado el servicio.

Seguramente nuestro modelo encuentre algunas relaciones que no podemos ver a simple vista. 

In [0]:
# Hacemos una pequeña exploración de los datos
display(ecomerceDF.describe())

summary,Customer ID,Product Category,Product Price,Quantity,Total Purchase Amount,Payment Method,Customer Age,Returns,Customer Name,Age,Gender,Churn
count,250000,250000,250000,250000,250000,250000,250000,202404,250000,250000,250000,250000
mean,25004.03624,null,254.659512,2.998896,2725.370732,null,43.940528,0.4978607142151341,null,43.940528,null,0.199496
stddev,14428.279590495233,null,141.5685768331717,1.4146938844213424,1442.9335650842972,null,15.350246304260477,0.49999665858211206,null,15.350246304260477,null,0.39962230265081483
min,1,Books,10,1,100,Cash,18,0.0,Aaron Acosta,18,Female,0
max,50000,Home,500,5,5350,PayPal,70,1.0,Zoe Lucero,70,Male,1


In [0]:

display(ecomerceDF.groupBy('Churn').count())

Churn,count
1,49874
0,200126


In [0]:
display(ecomerceDF.groupBy('Churn', 'Gender').count())

Churn,Gender,count
1,Male,25234
0,Male,99206
0,Female,100920
1,Female,24640


In [0]:
%sql

SELECT Churn, Gender, count(1) as count FROM ecomerce group by Churn, Gender order by Gender

Churn,Gender,count
0,Female,100920
1,Female,24640
0,Male,99206
1,Male,25234


Databricks visualization. Run in Databricks to view.

In [0]:
%sql

SELECT Churn, Product_Category, count(1) as count FROM ecomerce group by Churn, Product_Category order by Product_Category

Churn,Product_Category,count
1,Books,14926
0,Books,59986
1,Clothing,14898
0,Clothing,60154
1,Electronics,9979
0,Electronics,40206
0,Home,39780
1,Home,10071


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Exploración de datos, sólo válida para analizar visulmente. (LIMIT 10 para la presentación del notebook en formato ynpb)
SELECT Churn, Age, count(1) as count FROM ecomerce group by Churn, Age order by Age LIMIT 10

Churn,Age,count
1,18,1131
0,18,3872
0,19,3744
1,19,996
0,20,3679
1,20,1013
0,21,3930
1,21,1059
1,22,1054
0,22,3729


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Exploración de datos, sólo válida para analizar visulmente. (LIMIT 10 para la presentación del notebook en formato ynpb)
SELECT Churn, Customer_Id, sum(Returns) as total_returns FROM ecomerce group by Churn, Customer_Id order by total_returns desc LIMIT 10

Churn,Customer_Id,total_returns
0,42157,10.0
0,26273,10.0
0,20762,9.0
0,15489,9.0
0,48872,9.0
0,13603,9.0
1,15597,9.0
0,25485,9.0
0,40830,9.0
0,48820,9.0


Databricks visualization. Run in Databricks to view.

In [0]:
%sql 
SELECT * FROM ECOMERCE WHERE CUSTOMER_ID = 26273

Customer_ID,Purchase_Date,Product_Category,Product_Price,Quantity,Total_Purchase_Amount,Payment_Method,Customer_Age,Returns,Customer_Name,Age,Gender,Churn
26273,2022-02-23T01:31:27.000Z,Home,373,4,3347,Credit Card,35,1.0,Gail Thompson,35,Female,0
26273,2020-12-11T04:10:51.000Z,Clothing,445,1,3965,Crypto,35,1.0,Gail Thompson,35,Female,0
26273,2023-07-15T03:13:38.000Z,Books,68,5,4949,Crypto,35,1.0,Gail Thompson,35,Female,0
26273,2021-12-17T05:55:21.000Z,Books,104,1,2470,Credit Card,35,1.0,Gail Thompson,35,Female,0
26273,2023-08-17T20:37:14.000Z,Home,18,5,2047,PayPal,35,1.0,Gail Thompson,35,Female,0
26273,2021-05-16T18:55:11.000Z,Home,128,1,2421,Credit Card,35,1.0,Gail Thompson,35,Female,0
26273,2020-07-08T19:18:22.000Z,Home,41,4,3824,PayPal,35,1.0,Gail Thompson,35,Female,0
26273,2023-01-03T09:59:22.000Z,Electronics,316,5,966,Cash,35,1.0,Gail Thompson,35,Female,0
26273,2023-01-24T02:17:51.000Z,Clothing,280,4,3390,Credit Card,35,0.0,Gail Thompson,35,Female,0
26273,2022-08-09T13:26:50.000Z,Books,413,2,1666,Credit Card,35,1.0,Gail Thompson,35,Female,0


# Elección del modelo y preparación de los datos
Se hace un tratamiento de los valores nulos. Se define la variable objetivo y las variables explicativas. Se eligen los algoritmos de entrenamiento supervisado y se trasforman las variables para que tengan el formato requerido por el algoritmo de entrenamiento.

In [0]:
# Miramos el total de registros para luego compararlo con el total eliminando los nulos 
print(ecomerceDF.count())

250000


In [0]:
# Eliminamos los valores nulos
ecomerceDF = ecomerceDF.na.drop()
print(ecomerceDF.count())

202404


In [0]:
from pyspark.sql import functions as F

# Calcular el tiempo entre compras para cada cliente
ecomerceDF = ecomerceDF.withColumn(
    "days_since_last_purchase",
    F.datediff(F.current_date(), F.col("Purchase Date"))
)

In [0]:
from pyspark.sql.functions import col, year, month, dayofmonth, dayofweek, hour
# Vamos a dividir la fecha para obtener valor de ella
ecomerceDF = ecomerceDF.withColumn("Year", year(col("Purchase Date"))) \
                        .withColumn("Month", month(col("Purchase Date"))) \
                        .withColumn("Day", dayofmonth(col("Purchase Date"))) \
                        .withColumn("DayOfWeek", dayofweek(col("Purchase Date"))) \
                        .withColumn("Hour", hour(col("Purchase Date")))

In [0]:
# Vamos a eliminar campos que puede que no afecten demasiado al modelo, como "Customer ID","Customer Age", 
# "Customer Name", "Purchase Date", "Customer_Age", dejando Age y el campo original de fecha 

# Eliminar columnas
ecomerceDF_clean = ecomerceDF.drop("Customer ID","Customer Age", "Customer Name", "Purchase Date")

# Ver las primeras filas para comprobar los cambios
ecomerceDF_clean.show(5)


+----------------+-------------+--------+---------------------+--------------+-------+---+------+-----+------------------------+----+-----+---+---------+----+
|Product Category|Product Price|Quantity|Total Purchase Amount|Payment Method|Returns|Age|Gender|Churn|days_since_last_purchase|Year|Month|Day|DayOfWeek|Hour|
+----------------+-------------+--------+---------------------+--------------+-------+---+------+-----+------------------------+----+-----+---+---------+----+
|     Electronics|           12|       3|                  740|   Credit Card|    0.0| 37|  Male|    0|                    2045|2020|    9|  8|        3|   9|
|            Home|          468|       4|                 2739|        PayPal|    0.0| 37|  Male|    0|                    1502|2022|    3|  5|        7|  12|
|            Home|          288|       2|                 3196|        PayPal|    0.0| 37|  Male|    0|                    1423|2022|    5| 23|        2|  18|
|        Clothing|          196|       1|     

In [0]:
from pyspark.sql.functions import udf, col, min, max
from pyspark.sql.types import ArrayType, FloatType

# Columnas a normalizar
cols_to_scale = ["Product Price", "Total Purchase Amount", "Quantity", "days_since_last_purchase"]

# =========================
# 1. Calcular min y max por columna
# =========================
stats = {}

for c in cols_to_scale:
    row = ecomerceDF_clean.agg(
        min(col(c)).alias("min"),
        max(col(c)).alias("max")
    ).collect()[0]
    
    stats[c] = (row["min"], row["max"])

# =========================
# 2. Aplicar Min-Max scaling manual
# =========================
ecomerceDF_transform = ecomerceDF_clean

for c in cols_to_scale:
    min_val, max_val = stats[c]
    
    ecomerceDF_transform = ecomerceDF_transform.withColumn(
        c + "_scaled",
        (col(c) - min_val) / (max_val - min_val)
    )

# =========================
# 3. Crear vector equivalente a "features_scaled"
# =========================
from pyspark.sql.functions import array

ecomerceDF_transform = ecomerceDF_transform.withColumn(
    "features_scaled",
    array(*[col(c + "_scaled") for c in cols_to_scale])
)

# =========================
# 4. UDF 
# =========================
def extract_features(vector):
    return vector

extract_udf = udf(extract_features, ArrayType(FloatType()))

ecomerceDF_transform = ecomerceDF_transform.withColumn(
    "scaled_values",
    col("features_scaled")
)

# =========================
# 5. Separar columnas normalizadas 
# =========================
ecomerceDF_normalized = ecomerceDF_transform

for i, name in enumerate(cols_to_scale):
    ecomerceDF_normalized = ecomerceDF_normalized.withColumn(
        name + "_scaled",
        col("scaled_values")[i]
    )

# =========================
# 6. Drop final 
# =========================
ecomerceDF_normalized = ecomerceDF_normalized.drop(
    "features_scaled",
    "scaled_values",
    "Product Price",
    "Quantity",
    "Total Purchase Amount",
    "days_since_last_purchase"
)

# =========================
# 7. Mostrar resultado
# =========================
ecomerceDF_normalized.show(5)

+----------------+--------------+-------+---+------+-----+----+-----+---+---------+----+--------------------+----------------------------+---------------+-------------------------------+
|Product Category|Payment Method|Returns|Age|Gender|Churn|Year|Month|Day|DayOfWeek|Hour|Product Price_scaled|Total Purchase Amount_scaled|Quantity_scaled|days_since_last_purchase_scaled|
+----------------+--------------+-------+---+------+-----+----+-----+---+---------+----+--------------------+----------------------------+---------------+-------------------------------+
|     Electronics|   Credit Card|    0.0| 37|  Male|    0|2020|    9|  8|        3|   9|0.004081632653061225|          0.1217374738045342|            0.5|             0.8144863266814486|
|            Home|        PayPal|    0.0| 37|  Male|    0|2022|    3|  5|        7|  12|  0.9346938775510204|          0.5025719184606592|           0.75|             0.4131559497413156|
|            Home|        PayPal|    0.0| 37|  Male|    0|2022|  

In [0]:
from pyspark.sql.functions import col, when, array
from pyspark.sql import functions as F

ecomerceDF_encoded = ecomerceDF_normalized

ecomerceDF_encoded = ecomerceDF_encoded.withColumn(
    "GenderIndex",
    when(col("Gender") == "Male", 1).otherwise(0)
)

ecomerceDF_encoded = ecomerceDF_encoded.withColumn(
    "GenderVec",
    F.array(col("GenderIndex"), (1 - col("GenderIndex")))
)

categories = [r[0] for r in ecomerceDF_encoded.select("Product Category").distinct().collect()]

for i, cat in enumerate(categories):
    ecomerceDF_encoded = ecomerceDF_encoded.withColumn(
        f"ProductCategory_{i}",
        when(col("Product Category") == cat, 1).otherwise(0)
    )

ecomerceDF_encoded = ecomerceDF_encoded.withColumn(
    "ProductCategoryVec",
    F.array(*[col(f"ProductCategory_{i}") for i in range(len(categories))])
)

methods = [r[0] for r in ecomerceDF_encoded.select("Payment Method").distinct().collect()]

for i, m in enumerate(methods):
    ecomerceDF_encoded = ecomerceDF_encoded.withColumn(
        f"PaymentMethod_{i}",
        when(col("Payment Method") == m, 1).otherwise(0)
    )

ecomerceDF_encoded = ecomerceDF_encoded.withColumn(
    "PaymentMethodVec",
    F.array(*[col(f"PaymentMethod_{i}") for i in range(len(methods))])
)

assembler_input = [
    "ProductCategoryVec",
    "PaymentMethodVec",
    "Total Purchase Amount_scaled",
    "days_since_last_purchase_scaled",
    "Returns"
]

#ecomerceDF_final = ecomerceDF_encoded.withColumn(
#    "features",
#    array(*assembler_input)
#)


# Aplanar features correctamente (sin arrays dentro de arrays)
feature_cols = (
    [f"ProductCategory_{i}" for i in range(len(categories))] +
    [f"PaymentMethod_{i}" for i in range(len(methods))] +
    [
        "Total Purchase Amount_scaled",
        "days_since_last_purchase_scaled",
        "Returns"
    ]
)

ecomerceDF_final = ecomerceDF_encoded.withColumn(
    "features",
    array(*[col(c).cast("double") for c in feature_cols])
)

In [0]:
df_ml = ecomerceDF_final.select("features", "Churn").toPandas()

import numpy as np

X = np.vstack(df_ml["features"].values)
y = df_ml["Churn"]



# Entrenamiento
Se divide el `dataset` en conjunto de datos de entrenamiento y de test y realiza el entrenamiento.

In [0]:
# Dividimos el dataset en datos de entrenamiento y de test
train_data, test_data = ecomerceDF_final.randomSplit([0.7, 0.3])

#Obtenemos las características y la predicción de los datos de entrenamiento y de test y lo pasamos a pandas para su gestión
train_pd = train_data.select("features", "Churn").toPandas()
test_pd = test_data.select("features", "Churn").toPandas()


In [0]:
import numpy as np

X_train = np.vstack(train_pd["features"].values)
y_train = train_pd["Churn"]

X_test = np.vstack(test_pd["features"].values)
y_test = test_pd["Churn"]

In [0]:
# Definimos una función que va a evaluar varios modelos
# Tendremos como parámetros el nombre del modelo, la definición del modelo, los datos de entrenamiento y de test
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    # Entrenar
    model.fit(X_train, y_train)
    
    # Predecir
    y_pred = model.predict(X_test)
    
    # Métricas
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    
    print(f"\n===== {name} =====")
    print("Accuracy:", acc)
    print("Confusion Matrix:\n", cm)
    print("Classification Report:\n", classification_report(y_test, y_pred))
    
    return acc, cm

In [0]:
# Definimos los modelos que vamos a usar 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Logistic Regression (Balanced)": LogisticRegression(class_weight='balanced', max_iter=1000),
    "Random Forest": RandomForestClassifier(class_weight='balanced')
}

In [0]:
# Recorremos los modelos y evaluamos cada uno de ellos
results = {}

for name, model in models.items():
    acc, cm = evaluate_model(name, model, X_train, y_train, X_test, y_test)
    results[name] = acc


===== Logistic Regression =====
Accuracy: 0.8013968129057644
Confusion Matrix:
 [[48882     0]
 [12114     0]]
Classification Report:
               precision    recall  f1-score   support

           0       0.80      1.00      0.89     48882
           1       0.00      0.00      0.00     12114

    accuracy                           0.80     60996
   macro avg       0.40      0.50      0.44     60996
weighted avg       0.64      0.80      0.71     60996



/databricks/python/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/databricks/python/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/databricks/python/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



===== Logistic Regression (Balanced) =====
Accuracy: 0.5249196668634009
Confusion Matrix:
 [[26403 22479]
 [ 6499  5615]]
Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.54      0.65     48882
           1       0.20      0.46      0.28     12114

    accuracy                           0.52     60996
   macro avg       0.50      0.50      0.46     60996
weighted avg       0.68      0.52      0.57     60996


===== Random Forest =====
Accuracy: 0.7669355367565086
Confusion Matrix:
 [[46106  2776]
 [11440   674]]
Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.94      0.87     48882
           1       0.20      0.06      0.09     12114

    accuracy                           0.77     60996
   macro avg       0.50      0.50      0.48     60996
weighted avg       0.68      0.77      0.71     60996



# Evaluación

Los modelos muestran un comportamiento bastante diferente cuando trabajamos con conjuntos de datos no balanceados.

El algoritmo de **Regresión logística** alcanzó la mayor precisión (80% aproximadamente), pero no logra identificar bien los casos de abandono (recall = 0). Esto nos indica que el modelo está sesgado hacia la clase mayoritaria.

El algoritmo de **Regresión logística con balanceo** de clases redujo la precisión general bastante (52% aproximadamente), pero mejoró de manera significativa el recall para la clase de abandono (46% aprox.) que es realmente lo que nos interesa . Esto la hace más adecuada para el objetivo comercial, ya que nos ayuda a identificar a los clientes con mayor probabilidad de abandonar el servicio siendo esto más importante que la precisión general.

El algoritmo **Random Forest** alcanzó una precisión relativamente alta (~77%), pero, al igual que la regresión logística, tuvo un rendimiento bastante malo en la detección de abandono (recall ≈ 6%).


# Conclusiones

Después de analizar los datos, se ve que aunque algunos modelos llegan al 80% de precisión, esa cifra engaña un poco porque los datos están muy desbalanceados.

Básicamente, la regresión logística normal falla porque casi siempre predice que los clientes se van a quedar (la clase que más hay). Al final, no detecta ningún caso de abandono (churn), lo que hace que el modelo no sirva de mucho para el negocio.
Cuando ajusté el modelo con class_weight='balanced', la cosa mejoró bastante: el recall subió al 46%, lo que significa que ya detectamos a casi la mitad de los que se van. Es verdad que la precisión general baja un poco, pero es lo normal cuando los datos están así de disparejos.

Por su parte, el Random Forest tiene una precisión alta, pero sigue sin ser bueno detectando el churn. Esto me hace pensar que el problema no es solo el modelo, sino los datos mismos. 

En cuanto a las variables, añadí manualmente "days_since_last_purchase", calculando los días desde la última compra, y ayudó a entender mejor cómo se comportan los clientes, aunque no arregló el problema del todo.

En conclusión, para la empresa es mucho mejor detectar a un cliente que se va a ir, aunque nos equivoquemos un poco más en general. Por eso, creo que la regresión logística balanceada es la mejor opción, porque es la que mejor equilibra el encontrar a los que abandonan sin perder demasiada precisión.

# Valoración personal
Este ejercicio ha permitido comprender la importancia del tratamiento del desbalanceo de clases y cómo métricas como la accuracy pueden resultar engañosas en problemas reales.